# APIM ❤️ Microsoft Web IQ

## Track Microsoft Web IQ usage with Azure API Management

Route [Microsoft Web IQ](https://webiq.microsoft.ai/documentation/overview/) Web Search requests through Azure API Management (APIM), block Browse at the gateway, keep the upstream API key out of client applications, and attribute request volume and latency to APIM subscriptions.

**Audience:** Developers and platform teams operating web-grounded applications.

**Prerequisites:**

- Microsoft Web IQ limited-access approval and an API key from Web IQ Profile Management.
- Python 3.12+, the repository environment installed with `uv sync`, and VS Code with the Jupyter extension.
- Azure CLI installed and authenticated.
- An Azure subscription with Contributor + RBAC Administrator, or Owner, permissions.

**Learning goals:**

- Secure a Web IQ credential in an APIM secret named value.
- Use APIM subscriptions as consumer identities.
- Block the Browse operation before it reaches Web IQ.
- Optionally authenticate Web IQ with a Microsoft Entra ID app-only token.
- Emit and query privacy-conscious usage and latency metrics in Application Insights.

```mermaid
flowchart LR
    Client[Client application]
    MicrosoftEntra[Microsoft Entra ID]
    subgraph Gateway[Azure API Management]
        Subscription[Validate APIM subscription]
        RequestMetric[Emit request metric]
        Operation{Operation?}
        BlockMetric[Emit blocked-request metric]
        Forbidden[Return structured 403]
        Auth{Web IQ bearer token present?}
        NamedValue[(Secret named value)]
        ApiKey[Inject x-apikey<br/>from secret named value]
        Entra[Pass through Entra bearer token]
        ResponseMetric[Emit response and latency metrics]
        ErrorMetric[Emit gateway-error metric]
    end
    Client -.->|Client credentials<br/>Web IQ scope| MicrosoftEntra
    MicrosoftEntra -.->|Bearer token| Client
    Client -->|APIM subscription key<br/>optional Web IQ bearer token| Subscription
    Subscription --> RequestMetric --> Operation
    Operation -->|Browse| BlockMetric --> Forbidden --> Client
    Operation -->|Web Search| Auth
    NamedValue --> ApiKey
    Auth -->|No| ApiKey --> WebIQ[Microsoft Web IQ]
    Auth -->|Yes| Entra --> WebIQ
    WebIQ --> ResponseMetric --> Client
    Subscription -.->|Policy or gateway failure| ErrorMetric
    ErrorMetric --> Client
    RequestMetric -.-> AppInsights[Application Insights]
    BlockMetric -.-> AppInsights
    ResponseMetric -.-> AppInsights
    ErrorMetric -.-> AppInsights
    AppInsights --> Logs[Log Analytics]
```

> `params.json` contains your Web IQ API key while deploying. It is ignored by this repository; do not share or commit it.


## Outline

1. Configure the lab and verify Azure CLI access.
2. Deploy APIM, Application Insights, and Log Analytics.
3. Send Web IQ requests through APIM subscriptions.
4. Optionally call Web IQ with Microsoft Entra ID.
5. Query request, response, and latency metrics.
6. Verify that APIM blocks Browse and review production considerations.


<a id='initialize'></a>
### 0️⃣ Initialize notebook variables

The Web IQ key is read from `WEBIQ_API_KEY` when available; otherwise the notebook prompts without echoing it. APIM clients receive separate subscription keys after deployment.


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from getpass import getpass

sys.path.insert(1, '../../shared')
import utils

deployment_name = 'web-iq'
resource_group_name = f'lab-{deployment_name}'
resource_group_location = 'westus2'

apim_sku = 'Basicv2'
web_iq_api_path = 'web-iq'
apim_subscriptions_config = [
    {'name': 'research-team', 'displayName': 'Research Team'},
    {'name': 'support-team', 'displayName': 'Support Team'},
]

web_iq_api_key = os.getenv('WEBIQ_API_KEY') or getpass('Microsoft Web IQ API key: ')
if not web_iq_api_key:
    raise ValueError('Set WEBIQ_API_KEY or enter a Web IQ API key when prompted.')

utils.print_ok('Notebook initialized')


<a id='azure-cli'></a>
### 1️⃣ Verify Azure CLI and the active subscription

Confirm that subsequent deployment commands target the intended tenant and subscription.


In [ ]:
output = utils.run('az account show', 'Retrieved Azure account', 'Failed to get the current Azure account')

if not output.success or not output.json_data:
    raise RuntimeError('Authenticate with Azure CLI by running az login, then rerun this cell.')

current_user = output.json_data['user']['name']
tenant_id = output.json_data['tenantId']
subscription_id = output.json_data['id']
utils.print_info(f'Current user: {current_user}')
utils.print_info(f'Tenant ID: {tenant_id}')
utils.print_info(f'Subscription ID: {subscription_id}')


<a id='deploy'></a>
### 2️⃣ Deploy the lab with Bicep

The deployment creates APIM, two APIM subscriptions, Application Insights, and Log Analytics. The Web IQ key becomes a secret APIM named value and is never returned as a deployment output. The API policy permits Web Search and returns `403 Forbidden` for Browse.


In [ ]:
utils.create_resource_group(resource_group_name, resource_group_location)

bicep_parameters = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'apimSku': {'value': apim_sku},
        'apimSubscriptionsConfig': {'value': apim_subscriptions_config},
        'webIqApiPath': {'value': web_iq_api_path},
        'webIqApiKey': {'value': web_iq_api_key},
    },
}

with open('params.json', 'w', encoding='utf-8') as parameters_file:
    json.dump(bicep_parameters, parameters_file)

output = utils.run(
    f'az deployment group create --name {deployment_name} --resource-group {resource_group_name} '
    '--template-file main.bicep --parameters params.json',
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed",
)
if not output.success:
    raise RuntimeError('Deployment failed. Review the Azure CLI output above.')


<a id='outputs'></a>
### 3️⃣ Retrieve gateway details

Only the APIM subscription keys are displayed, masked to their final four characters. The upstream Web IQ key remains inside APIM.


In [ ]:
output = utils.run(
    f'az deployment group show --name {deployment_name} --resource-group {resource_group_name}',
    f"Retrieved deployment '{deployment_name}'",
    f"Failed to retrieve deployment '{deployment_name}'",
)
if not output.success or not output.json_data:
    raise RuntimeError('Could not retrieve the deployment outputs.')

apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM gateway URL')
application_insights_name = utils.get_deployment_output(output, 'applicationInsightsName', 'Application Insights name')
web_iq_api_path = utils.get_deployment_output(output, 'webIqApiPath', 'Web IQ API path')
apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("'", '"'))

for subscription in apim_subscriptions:
    utils.print_info(f"{subscription['displayName']}: ****{subscription['key'][-4:]}")


<a id='search'></a>
### 4️⃣ Send Web Search requests through APIM

Each team uses its own APIM subscription key. APIM replaces that client credential with the secret `x-apikey` required by Web IQ, forwards the request, and emits usage metrics. Only a compact result summary is printed.


In [ ]:
import requests

search_url = f'{apim_resource_gateway_url}/{web_iq_api_path}/search/web'
workloads = [
    ('research-team', 'How does Azure API Management support AI gateways?'),
    ('support-team', 'What is Microsoft Web IQ?'),
]
subscription_by_name = {item['name']: item for item in apim_subscriptions}
request_summary = []

for subscription_name, query in workloads:
    subscription = subscription_by_name[subscription_name]
    started = time.perf_counter()
    response = requests.post(
        search_url,
        headers={
            'Ocp-Apim-Subscription-Key': subscription['key'],
            'content-type': 'application/json',
        },
        json={
            'query': query,
            'maxResults': 3,
            'maxLength': 3000,
            'contentFormat': 'markdown',
        },
        timeout=30,
    )
    elapsed_ms = (time.perf_counter() - started) * 1000
    utils.print_response_code(response)

    try:
        data = response.json()
    except requests.JSONDecodeError:
        data = {'rawResponse': response.text[:500]}

    results = data.get('webResults', []) if isinstance(data, dict) else []
    request_summary.append({
        'subscription': subscription_name,
        'status': response.status_code,
        'latency_ms': round(elapsed_ms, 1),
        'results': len(results),
        'trace_id': data.get('traceId') if isinstance(data, dict) else None,
    })

    for result in results[:3]:
        print(f"- {result.get('title', '(untitled)')}: {result.get('url', '')}")
    if not response.ok:
        print(json.dumps(data, indent=2)[:2000])

request_summary


<a id='entra-id'></a>
### 5️⃣ Optional: authenticate Web IQ with Microsoft Entra ID

Web IQ recommends [Entra ID app-only authentication](https://webiq.microsoft.ai/documentation/authentication/#entra-id) for production workloads. Create an app registration and client credential, then bind its **Application (client) ID** in Web IQ Profile Management. Set `WEBIQ_TENANT_ID`, `WEBIQ_CLIENT_ID`, and `WEBIQ_CLIENT_SECRET` in your environment before running this cell. The token scope is `https://api.microsoft.ai/.default`.

The APIM subscription key still identifies the consuming team. When APIM sees the bearer token, it removes `x-apikey` and lets Web IQ validate the Entra token. If the environment variables are absent, this optional step is skipped.


In [ ]:
from msal import ConfidentialClientApplication

entra_settings = {
    'tenant_id': os.getenv('WEBIQ_TENANT_ID'),
    'client_id': os.getenv('WEBIQ_CLIENT_ID'),
    'client_secret': os.getenv('WEBIQ_CLIENT_SECRET'),
}

if not all(entra_settings.values()):
    utils.print_info(
        'Optional Entra ID call skipped. Set WEBIQ_TENANT_ID, WEBIQ_CLIENT_ID, and WEBIQ_CLIENT_SECRET to run it.'
    )
else:
    entra_client = ConfidentialClientApplication(
        client_id=entra_settings['client_id'],
        client_credential=entra_settings['client_secret'],
        authority=f"https://login.microsoftonline.com/{entra_settings['tenant_id']}",
    )
    token_result = entra_client.acquire_token_for_client(
        scopes=['https://api.microsoft.ai/.default']
    )
    if 'access_token' not in token_result:
        raise RuntimeError(token_result.get('error_description', 'Could not acquire a Web IQ access token.'))

    entra_response = requests.post(
        search_url,
        headers={
            'Ocp-Apim-Subscription-Key': apim_subscriptions[0]['key'],
            'Authorization': f"Bearer {token_result['access_token']}",
            'content-type': 'application/json',
        },
        json={
            'query': 'What is Microsoft Web IQ?',
            'maxResults': 3,
            'maxLength': 3000,
            'contentFormat': 'markdown',
        },
        timeout=30,
    )
    utils.print_response_code(entra_response)
    entra_data = entra_response.json()
    print({
        'traceId': entra_data.get('traceId'),
        'resultCount': len(entra_data.get('webResults', [])),
    })


<a id='metrics'></a>
### 6️⃣ Query usage metrics in Application Insights

Custom metrics can take several minutes to arrive. If the tables are empty, wait briefly and rerun these cells. The first query reports request volume and blocked calls by APIM subscription and operation.


In [ ]:
import pandas as pd

def query_application_insights(kql: str) -> pd.DataFrame:
    result = utils.run(
        f'az monitor app-insights query --app {application_insights_name} '
        f'--resource-group {resource_group_name} --analytics-query {json.dumps(kql)}',
        'Application Insights query succeeded',
        'Application Insights query failed',
    )
    if not result.success or not result.json_data.get('tables'):
        return pd.DataFrame()
    table = result.json_data['tables'][0]
    return pd.DataFrame(
        table.get('rows', []),
        columns=[column['name'] for column in table.get('columns', [])],
    )

usage_query = r'''
customMetrics
| where timestamp > ago(1h) and name in ('Web IQ Requests', 'Web IQ Blocked Requests')
| extend dimensions = todynamic(customDimensions)
| extend SubscriptionId = tostring(dimensions['Subscription ID'])
| extend OperationId = tostring(dimensions['Operation ID'])
| extend Authentication = tostring(dimensions['Authentication'])
| summarize Requests = sumif(value, name == 'Web IQ Requests'), BlockedRequests = sumif(value, name == 'Web IQ Blocked Requests')
    by SubscriptionId, OperationId, Authentication
| order by Requests desc
'''

usage_df = query_application_insights(usage_query)
usage_df


The latency query groups completed calls by consumer, operation, and upstream HTTP status. This makes throttling and service errors visible without capturing request or response bodies.


In [ ]:
latency_query = r'''
customMetrics
| where timestamp > ago(1h) and name == 'Web IQ Latency'
| extend dimensions = todynamic(customDimensions)
| extend SubscriptionId = tostring(dimensions['Subscription ID'])
| extend OperationId = tostring(dimensions['Operation ID'])
| extend Authentication = tostring(dimensions['Authentication'])
| extend StatusCode = tostring(dimensions['Status Code'])
| summarize Calls = count(), AverageLatencyMs = round(avg(value), 1), P95LatencyMs = round(percentile(value, 95), 1)
    by SubscriptionId, OperationId, Authentication, StatusCode
| order by Calls desc
'''

latency_df = query_application_insights(latency_query)
latency_df


### Exercise — verify that Browse is blocked

Use the second APIM subscription to call `/browse`. Before running the next cell, predict the status and response body. The request should return APIM's structured `403` without reaching Web IQ. After telemetry arrives, rerun the usage query above to see `browse-url` with one blocked request.


In [ ]:
# Answer scaffold: change target_url to verify that every Browse target is blocked.
def browse_with_subscription(target_url: str, subscription_index: int = 1) -> requests.Response:
    subscription = apim_subscriptions[subscription_index]
    response = requests.post(
        f'{apim_resource_gateway_url}/{web_iq_api_path}/browse',
        headers={
            'Ocp-Apim-Subscription-Key': subscription['key'],
            'content-type': 'application/json',
        },
        json={
            'url': target_url,
            'contentFormat': 'markdown',
            'maxLength': 3000,
            'liveCrawl': 'fallback',
        },
        timeout=30,
    )
    utils.print_response_code(response)
    print(json.dumps(response.json(), indent=2))
    return response

browse_response = browse_with_subscription('https://news.microsoft.com/source/')
assert browse_response.status_code == 403
assert browse_response.json()['errorCode'] == 'BrowseOperationBlocked'


### Pitfalls and extensions

- **Telemetry delay:** Application Insights custom metrics are not immediate; rerun the queries after a few minutes.
- **Credential boundaries:** Clients send only APIM subscription keys. Never forward those as `x-apikey`, and never return the Web IQ named value.
- **Metric cardinality:** Keep dimensions bounded. Search text, URLs, trace IDs, and client IP addresses are intentionally excluded.
- **Policy scope:** The block checks APIM's stable operation ID (`browse-url`), not a client-controlled URL string. Remove or revise the `choose` block in [policy.xml](policy.xml) only when Browse is approved for your consumers.
- **Two kinds of instrumentation:** These APIM metrics measure gateway usage. Web IQ [instrumentation](https://webiq.microsoft.ai/documentation/instrumentation/) separately records which citations an LLM uses and which links users click.
- **Extension:** Add APIM rate limits per subscription, or proxy other Web IQ verticals after confirming that your Web IQ account permits them.


<a id='portal'></a>
### View metrics in the Azure portal

Open the deployed Application Insights resource, select **Metrics**, choose the `web-iq` custom namespace, and select **Web IQ Requests**, **Web IQ Blocked Requests**, or **Web IQ Latency**. Split by **Subscription ID**, **Operation ID**, **Authentication**, or **Status Code**.


<a id='clean-up'></a>
### 🗑️ Clean up resources

Run [clean-up-resources.ipynb](clean-up-resources.ipynb) when finished to remove the resource group and avoid further charges.
